In [259]:
from ast import parse
from calendar import day_abbr
from unittest.mock import inplace

import numpy as np
import pandas as pd
from numpy import dtype
from pygments.lexers import q
from tornado.http1connection import parse_int

# 导入数据

In [260]:
# 导入数据
df = pd.read_csv('data/employees.csv')

# pandas 中的 read_json比较弱，建议用 json 头文件
import json
with open('data/test.json') as f:
    data = json.load(f)
    # print(data) # 字典,键位'uers',值有多个
    # print(data['users'])
    df = pd.DataFrame(data['users']) # 将 data 转为 DataFrame 类型
    print(df)
    f.close()

   id name  age                 email  is_active   join_date
0   1   张三   28  zhangsan@example.com       True  2022-03-15
1   2   李四   35      lisi@example.com      False  2021-11-02
2   3   王五   24    wangwu@example.com       True  2023-01-20


In [261]:
# 缺失值的处理
# nan:not a number
s = pd.Series([1,2,np.nan,None,pd.NA])
print(s.isna())
print(s.isnull())
df = pd.DataFrame([[1,np.nan,None],[np.nan,1,3],[2,2,3]],columns=['第一列','第二列','第三列'])
print(df.isna().sum()) # 默认按列
print(df.isna().sum(axis=1))

0    False
1    False
2     True
3     True
4     True
dtype: bool
0    False
1    False
2     True
3     True
4     True
dtype: bool
第一列    1
第二列    1
第三列    1
dtype: int64
0    2
1    1
2    0
dtype: int64


#

In [262]:
# 剔除缺失值
print(df)
print('-'*40)
print(df.dropna()) # 只要有一行 NaN，那么会删除整行的结果
print('-'*40)
print(df.dropna(how='all')) # 只有当整行全为 NaN，才删除
print('-'*40)
# 如果想对列操作，加一个参数 axis=1 即可
print(df.dropna(axis=1))
print('-'*40)
# 如果想自定义量，也就是 thresh=n，代表有多少个 NaN 就删除整行
print(df.dropna(thresh=2))
print('-'*40)
# 指定索引 只要这一列上有 NaN，那么就直接删除这这一行
print(df.dropna(subset=['第一列']))

   第一列  第二列  第三列
0  1.0  NaN  NaN
1  NaN  1.0  3.0
2  2.0  2.0  3.0
----------------------------------------
   第一列  第二列  第三列
2  2.0  2.0  3.0
----------------------------------------
   第一列  第二列  第三列
0  1.0  NaN  NaN
1  NaN  1.0  3.0
2  2.0  2.0  3.0
----------------------------------------
Empty DataFrame
Columns: []
Index: [0, 1, 2]
----------------------------------------
   第一列  第二列  第三列
1  NaN  1.0  3.0
2  2.0  2.0  3.0
----------------------------------------
   第一列  第二列  第三列
0  1.0  NaN  NaN
2  2.0  2.0  3.0


In [263]:
# 填充缺失值
df = pd.read_csv('data/weather_withna.csv')
print(df.isna().sum(axis=0))
# 用字典填充
print(df.fillna({'temp_max':20, 'temp_min':5}).tail())
# 默认用平均值填充
print(df.fillna(df[['wind']].mean()).tail())
#用相邻前/后一个值填充
print(df.ffill().tail())
print(df.bfill().tail())

date               0
precipitation    303
temp_max         303
temp_min         303
wind             303
weather          303
dtype: int64
            date  precipitation  temp_max  temp_min  wind weather
1456  2015-12-27            NaN      20.0       5.0   NaN     NaN
1457  2015-12-28            NaN      20.0       5.0   NaN     NaN
1458  2015-12-29            NaN      20.0       5.0   NaN     NaN
1459  2015-12-30            NaN      20.0       5.0   NaN     NaN
1460  2015-12-31           20.6      12.2       5.0   3.8    rain
            date  precipitation  temp_max  temp_min      wind weather
1456  2015-12-27            NaN       NaN       NaN  3.242055     NaN
1457  2015-12-28            NaN       NaN       NaN  3.242055     NaN
1458  2015-12-29            NaN       NaN       NaN  3.242055     NaN
1459  2015-12-30            NaN       NaN       NaN  3.242055     NaN
1460  2015-12-31           20.6      12.2       5.0  3.800000    rain
            date  precipitation  temp_max  te

In [264]:
data = {
    "name":['a','a','b','c','c'],
    'age':[26,30,20,30,30],
    'city':['NY','PK','NC','QD','QD']
}
df = pd.DataFrame(data)
# 重复数据的处理
df.duplicated() #一整条都是一样的，返回 True
df.drop_duplicates(subset=['name']) # 指定列
df.drop_duplicates(subset=['name'],keep='last') # 指定列,保留最后一次出现的行

,name,age,city
1,a,30,PK
2,b,20,NC
4,c,30,QD


# 数据类型的转换

In [265]:
df = pd.read_csv('data/sleep.csv')

# 数据类型的转换
df['age'] = df['age'].astype('int16')
# 转换为分类变量，节省内存
df['gender'] = df['gender'].astype('category') # 分类变量 将 gender 中的值分为两类：Female、Male
df['is_male'] = df['gender'].map({'Male':True,'Female':False})
print(df.is_male) # Categories (2, bool): [False, True] 如果原来的 gender是分类变量，那么 map 映射过后也是分类变量

0       True
1      False
2       True
3       True
4       True
       ...  
395    False
396    False
397    False
398    False
399     True
Name: is_male, Length: 400, dtype: category
Categories (2, bool): [False, True]


# 数据变形

In [266]:
# 数据变形
data = {
    'name':['alice','bob','mike'],
    'math':np.random.randint(0,100,3),
    'english':np.random.randint(0,100,3),
    'physics':np.random.randint(0,100,3)
}
df = pd.DataFrame(data)
# 转置
print(df.T)

# 将短表转为长表转为宽表 —— melt
"""
alice math    a1
alice english a2
alice physics a3
"""
# id_vars: 保持不动的列（标识符）
# value_vars: 要被“融化”的列
df2 = df.melt(id_vars=['name'],var_name='科目',value_name='分数')
df2 = df2.sort_values(['name'])
# 将长表转为宽表 —— pivot
# index: 变形后，哪一列作为新的“行索引”。
# columns: 哪一列的取值变成新的“列名”。
# values: 哪些数据填充到格子里面
pd.pivot(df2,index='name',columns='科目',values='分数')

             0    1     2
name     alice  bob  mike
math        52   59    13
english     47   17    95
physics     89   34    38


科目,english,math,physics
name,,,
alice,47,52,89
bob,17,59,34
mike,95,13,38


In [267]:
# 分列
data = {
    'name':['Smith Tom','Smith Bob'],
    'math':[98,87],
    'english':[70,80]
}
df = pd.DataFrame(data)
# 按照 first name 和 last name 分类
df[['first name','last name']] = df['name'].str.split(' ',expand=True) # expand = True 代表制成表
df = pd.read_csv('data/sleep.csv')
df = df[['person_id','blood_pressure']]
df[['max_blood_pressure','min_blood_pressure']] = df['blood_pressure'].str.split('/',expand=True).astype('int16')
# df.info()
df.max_blood_pressure.mean()

np.float64(122.19)

# 数据分箱

In [268]:
# 数据分箱
df = pd.read_csv('data/employees.csv')

pd.cut(df['salary'],bins=3) # bins=n 代表把数据分为 n 段区间
# """
# 起始值：数据最小值
# 中间值：百分位数
# 最终值：数据最大值
# """
pd.cut(df['salary'],bins=3).value_counts()

df['收入范围']=pd.cut(df['salary'],bins=[0,10000,20000,30000],labels=['低','中','高'])


# 等频率分割，尽量让每个范围的 id 总数差不多
pd.qcut(df['salary'],3).value_counts()


salary
(2099.999, 3333.333]    36
(7966.667, 24000.0]     36
(3333.333, 7966.667]    35
Name: count, dtype: int64

In [269]:
df = pd.read_csv('data/sleep.csv')
df['睡眠质量'] = pd.cut(df['sleep_quality'],bins=3,labels=['低','中','高'])
df['gender'] = df['gender'].astype('category')
df['gender'].value_counts()

gender
Female    201
Male      199
Name: count, dtype: int64

# 时间数据的处理

In [352]:
d = pd.Timestamp('2007-8-5')
print(d)
print(d.year)

data = {
    'id':[0,1,2],
    'date':['2007-8-5','2007-8-6','2007-8-7']
}
df = pd.DataFrame(data)
df['dateTime'] = pd.to_datetime(df['date'])
# print(df['dateTime'].date_name()) 报错
# 应为实际上df['dateTime']类型是 Series，但是 date_name 作用的对象 datetime64
# 我们需要一个访问器 dt 用于转换，类似于前面分组 str.split()
print(df['dateTime'].dt.date)


2007-08-05 00:00:00
2007
0    2007-08-05
1    2007-08-06
2    2007-08-07
Name: dateTime, dtype: object


In [349]:
# df = pd.read_csv('data/weather.csv')
# df['datetime'] = pd.to_datetime(df['date'])
# df['datetime'].dt.day_name()
# 或者直接在创建时指定类型
df = pd.read_csv('data/weather.csv',parse_dates=['date'])
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1461 entries, 0 to 1460
Data columns (total 6 columns):
 #   Column         Non-Null Count  Dtype         
---  ------         --------------  -----         
 0   date           1461 non-null   datetime64[us]
 1   precipitation  1461 non-null   float64       
 2   temp_max       1461 non-null   float64       
 3   temp_min       1461 non-null   float64       
 4   wind           1461 non-null   float64       
 5   weather        1461 non-null   str           
dtypes: datetime64[us](1), float64(4), str(1)
memory usage: 68.6 KB


In [350]:
# 日期数据作为索引,我们就可以进行切片
# df2 = df.set_index('date')
# df.set_index('date',inplace=True) # inplace=True 就地修改，不创建新数组
# df.info()
print(df['2013-01':'2013-02'].tail())

TypeError: cannot do slice indexing on RangeIndex with these indexers [2013-01] of type str

In [246]:
d1 = pd.Timestamp('2007-08-05')
d2 = pd.Timestamp('2026-04-02')
d3 = d2 - d1
print(d3)
print(type(d3))

6815 days 00:00:00
<class 'pandas.Timedelta'>


In [258]:
# 创建时间序列
days = pd.date_range("2025-07-03","2026-04-02",freq='D')

days2 = pd.date_range("2025-07-05",periods=10,freq='D')

In [273]:
# 重新采样
df = pd.read_csv('data/weather.csv',parse_dates=['date'])
df.set_index('date',inplace=True)

In [275]:
df[ ["temp_max","temp_min"] ].resample('YE').mean()

,temp_max,temp_min
date,,
2012-12-31,15.276776,7.289617
2013-12-31,16.058904,8.153973
2014-12-31,16.995890,8.662466
2015-12-31,17.427945,8.835616


# 分组聚合

In [302]:
# 分组聚合
# df.groupby('分组的字段'['聚合的字段'])
df = pd.read_csv('data/employees.csv')
# df['department_id'].isna().sum()
df = df.dropna(subset=['department_id'])
df['department_id'] = df['department_id'].astype('int64')
# df.groupby('department_id').groups # 查看分组，字典
# print(df.groupby('department_id').get_group(20)) # 查看指定组
df2 = df.groupby('department_id')[['salary']].mean().round(2)
df2.sort_values('salary',ascending=False)

,salary
department_id,
90,19333.33
110,10150.00
70,10000.00
20,9500.00
80,8955.88
100,8600.00
40,6500.00
60,5760.00
10,4400.00


In [312]:
# 计算不同部门不同岗位人的平均薪资
df2 = df.groupby(['department_id','job_id'])['salary'].mean()
df2 = df2.reset_index()
df2.sort_values('salary',ascending=False)

,department_id,job_id,salary
13,90,AD_PRES,24000.000000
14,90,AD_VP,17000.000000
1,20,MK_MAN,13000.000000
11,80,SA_MAN,12200.000000
18,110,AC_MGR,12000.000000
16,100,FI_MGR,12000.000000
4,30,PU_MAN,11000.000000
10,70,PR_REP,10000.000000
12,80,SA_REP,8396.551724
17,110,AC_ACCOUNT,8300.000000


In [313]:
# 企鹅数据分析
import pandas as pd
import numpy as np

# 导入数据

In [317]:
df = pd.read_csv('data/penguins.csv')
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 344 entries, 0 to 343
Data columns (total 7 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   species            344 non-null    str    
 1   island             344 non-null    str    
 2   bill_length_mm     342 non-null    float64
 3   bill_depth_mm      342 non-null    float64
 4   flipper_length_mm  342 non-null    float64
 5   body_mass_g        342 non-null    float64
 6   sex                333 non-null    str    
dtypes: float64(4), str(3)
memory usage: 18.9 KB


In [323]:
# 数据的清洗
print(df.isna().sum())
df.dropna(inplace=True)
print(df.isna().sum())

species              0
island               0
bill_length_mm       0
bill_depth_mm        0
flipper_length_mm    0
body_mass_g          0
sex                  0
dtype: int64
species              0
island               0
bill_length_mm       0
bill_depth_mm        0
flipper_length_mm    0
body_mass_g          0
sex                  0
dtype: int64


In [331]:
# 数据特征的构造
df['sex'] = df['sex'].astype('category')
df['bill_ratio'] = df['bill_length_mm'] / df['bill_depth_mm']

In [336]:
# 数据分析
# 把体重分为低中高
labels = ['low','medium','high']
df['mass_level'] = pd.cut(df['body_mass_g'],bins=3,labels=labels)
print(df['mass_level'].value_counts())

df.groupby(['sex','island']).agg({'body_mass_g':['mean','count']})


mass_level
low       150
medium    128
high       55
Name: count, dtype: int64


body_mass_g      
                         mean count
sex    island                      
Female Biscoe     4319.375000    80
       Dream      3446.311475    61
       Torgersen  3395.833333    24
Male   Biscoe     5104.518072    83
       Dream      3987.096774    62
       Torgersen  4034.782609    23

In [354]:
df = pd.read_csv('data/sleep.csv')
df.info()
df

<class 'pandas.DataFrame'>
RangeIndex: 400 entries, 0 to 399
Data columns (total 13 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   person_id                400 non-null    int64  
 1   gender                   400 non-null    str    
 2   age                      400 non-null    int64  
 3   occupation               400 non-null    str    
 4   sleep_duration           400 non-null    float64
 5   sleep_quality            400 non-null    float64
 6   physical_activity_level  400 non-null    int64  
 7   stress_level             400 non-null    int64  
 8   bmi_category             400 non-null    str    
 9   blood_pressure           400 non-null    str    
 10  heart_rate               400 non-null    int64  
 11  daily_steps              400 non-null    int64  
 12  sleep_disorder           110 non-null    str    
dtypes: float64(2), int64(6), str(5)
memory usage: 40.8 KB


,person_id,gender,age,occupation,sleep_duration,sleep_quality,physical_activity_level,stress_level,bmi_category,blood_pressure,heart_rate,daily_steps,sleep_disorder
0,1,Male,29,Manual Labor,7.4,7.0,41,7,Obese,124/70,91,8539,NaN
1,2,Female,43,Retired,4.2,4.9,41,5,Obese,131/86,81,18754,NaN
2,3,Male,44,Retired,6.1,6.0,107,4,Underweight,122/70,81,2857,NaN
3,4,Male,29,Office Worker,8.3,10.0,20,10,Obese,124/72,55,6886,NaN
4,5,Male,67,Retired,9.1,9.5,19,4,Overweight,133/78,97,14945,Insomnia
...,...,...,...,...,...,...,...,...,...,...,...,...,...
395,396,Female,36,Student,4.5,7.9,73,7,Normal,118/66,64,14497,Sleep Apnea
396,397,Female,45,Manual Labor,6.0,6.1,72,8,Obese,132/80,65,12848,Insomnia
397,398,Female,30,Student,5.3,6.5,58,10,Obese,125/76,66,15255,Insomnia
398,399,Female,41,Retired,11.0,9.1,73,9,Obese,130/75,75,6567,Sleep Apnea


In [355]:
# 数据清洗
df.isna().sum()

person_id                    0
gender                       0
age                          0
occupation                   0
sleep_duration               0
sleep_quality                0
physical_activity_level      0
stress_level                 0
bmi_category                 0
blood_pressure               0
heart_rate                   0
daily_steps                  0
sleep_disorder             290
dtype: int64

In [358]:
df.dropna(axis="columns",inplace=True)
df.isna().sum()

person_id                  0
gender                     0
age                        0
occupation                 0
sleep_duration             0
sleep_quality              0
physical_activity_level    0
stress_level               0
bmi_category               0
blood_pressure             0
heart_rate                 0
daily_steps                0
dtype: int64

In [393]:
# 数据特征值的改造
df['gender'] = df['gender'].astype('category')
df['bmi_category'] = df['bmi_category'].astype('category')
df['occupation'] = df['occupation'].astype('category')
df[['high','low']] = df['blood_pressure'].str.split('/',expand=True)
# 睡眠质量的分箱
labels=['low','medium','high']
df['sleep_level'] = pd.cut(df['sleep_quality'],bins=3,labels=labels)
# 年龄的分箱
labels=['青年','中年','老年']
df['age_level'] = pd.cut(df['age'],bins=[0,18,50,200],labels=labels)

,person_id,gender,age,occupation,sleep_duration,sleep_quality,physical_activity_level,stress_level,bmi_category,blood_pressure,heart_rate,daily_steps,high,low,sleep_level,age_level
0,1,Male,29,Manual Labor,7.4,7.0,41,7,Obese,124/70,91,8539,124,70,medium,中年
1,2,Female,43,Retired,4.2,4.9,41,5,Obese,131/86,81,18754,131,86,medium,中年
2,3,Male,44,Retired,6.1,6.0,107,4,Underweight,122/70,81,2857,122,70,medium,中年
3,4,Male,29,Office Worker,8.3,10.0,20,10,Obese,124/72,55,6886,124,72,high,中年
4,5,Male,67,Retired,9.1,9.5,19,4,Overweight,133/78,97,14945,133,78,high,老年
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
395,396,Female,36,Student,4.5,7.9,73,7,Normal,118/66,64,14497,118,66,high,中年
396,397,Female,45,Manual Labor,6.0,6.1,72,8,Obese,132/80,65,12848,132,80,medium,中年
397,398,Female,30,Student,5.3,6.5,58,10,Obese,125/76,66,15255,125,76,medium,中年
398,399,Female,41,Retired,11.0,9.1,73,9,Obese,130/75,75,6567,130,75,high,中年


In [396]:
# 数据分析与统计
print(df.bmi_category.value_counts())
df.groupby(['age_level','bmi_category']).agg(person_sum=('person_id','count'),age_mean=('age','mean'))

bmi_category
Overweight     109
Underweight    102
Obese           98
Normal          91
Name: count, dtype: int64


person_sum   age_mean
age_level bmi_category                       
青年        Normal                 6  18.000000
          Obese                  5  18.000000
          Overweight             6  18.000000
          Underweight           11  18.000000
中年        Normal                62  35.903226
          Obese                 73  36.602740
          Overweight            78  36.166667
          Underweight           72  35.680556
老年        Normal                23  61.130435
          Obese                 20  58.050000
          Overweight            25  62.000000
          Underweight           19  56.368421